# Clase 214 — Parquet vs CSV vs Avro: benchmarks y schema evolution

Requiere: `pip install pyarrow polars duckdb fastavro`.

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq, polars as pl, pandas as pd, numpy as np, time, json
from pathlib import Path
import tempfile, shutil

WORK = Path(tempfile.gettempdir()) / 'parquet_bench'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()

rng = np.random.default_rng(42)
N = 2_000_000
df = pl.DataFrame({
    'zone_id': rng.integers(0, 100, N),
    'fare':    rng.uniform(5, 100, N),
    'tip':     rng.uniform(0, 20, N),
    'pickup_date': pl.date(2024, 1, 1) + pl.duration(days=pl.Series(rng.integers(0, 90, N))),
    'borough': rng.choice(['Manhattan', 'Brooklyn', 'Queens', 'Bronx'], N),
    'note':    rng.choice(['short ride', 'long ride', 'airport', 'rush hour', None], N),
})
print(f'dataset: {N:,} rows, {len(df.columns)} cols')

## 1. Benchmark: tamaño por formato

In [ ]:
results = []
df.write_csv(WORK / 'd.csv')
results.append(('CSV', (WORK / 'd.csv').stat().st_size))

for comp in ['snappy', 'zstd', 'gzip', 'lz4']:
    p = WORK / f'd_{comp}.parquet'
    df.write_parquet(p, compression=comp)
    results.append((f'Parquet/{comp}', p.stat().st_size))

import gzip
j = WORK / 'd.json.gz'
with gzip.open(j, 'wt') as f:
    for row in df.iter_rows(named=True):
        f.write(json.dumps(row, default=str) + '\n')
results.append(('JSONL/gzip', j.stat().st_size))

print(f'{"formato":18} {"MB":>8}  ratio vs CSV')
csv_size = next(s for n, s in results if n == 'CSV')
for name, size in results:
    print(f'{name:18} {size / 1024 / 1024:>8.2f}  {size / csv_size:.2%}')

## 2. Benchmark: tiempo de query analítica

In [ ]:
import duckdb
con = duckdb.connect()

def bench(name, sql):
    t0 = time.perf_counter()
    n = con.execute(sql).fetchone()[0]
    return name, (time.perf_counter() - t0) * 1000, n

csv_p = str(WORK / 'd.csv').replace(chr(92), '/')
snap_p = str(WORK / 'd_snappy.parquet').replace(chr(92), '/')
zstd_p = str(WORK / 'd_zstd.parquet').replace(chr(92), '/')

queries = [
    ('CSV',  f"SELECT COUNT(*) FROM '{csv_p}' WHERE borough='Manhattan'"),
    ('Parquet snappy', f"SELECT COUNT(*) FROM '{snap_p}' WHERE borough='Manhattan'"),
    ('Parquet zstd',   f"SELECT COUNT(*) FROM '{zstd_p}' WHERE borough='Manhattan'"),
]
for name, sql in queries:
    n, t, _ = bench(name, sql)
    print(f'{n:20} {t:>8.1f} ms')

# Column pruning: leer solo 1 columna
print('\n--- SELECT 1 column con filter ---')
for name, p in [('CSV', csv_p), ('Parquet snappy', snap_p)]:
    t0 = time.perf_counter()
    con.execute(f"SELECT AVG(fare) FROM '{p}' WHERE borough='Manhattan'").fetchone()
    print(f'{name:20} {(time.perf_counter() - t0) * 1000:>8.1f} ms')

## 3. Inspeccionar metadata Parquet

In [ ]:
meta = pq.ParquetFile(snap_p).metadata
print(f'archivo: {meta.num_rows:,} rows, {meta.num_row_groups} row groups, {meta.num_columns} columns')

rg = meta.row_group(0)
print(f'\nrow group 0: {rg.num_rows:,} rows')
for i in range(rg.num_columns):
    col = rg.column(i)
    s = col.statistics
    if s:
        print(f'  col {i} ({col.path_in_schema:12}): min={s.min}, max={s.max}, nulls={s.null_count}')

## 4. Avro: serialización + schema evolution

In [ ]:
import fastavro
from io import BytesIO

schema_v1 = {
    'type': 'record',
    'name': 'Click',
    'fields': [
        {'name': 'user_id', 'type': 'string'},
        {'name': 'page',    'type': 'string'},
        {'name': 'ts',      'type': 'double'},
    ],
}

schema_v2 = {  # backward compatible: agregar campo OPCIONAL con default
    'type': 'record',
    'name': 'Click',
    'fields': [
        {'name': 'user_id', 'type': 'string'},
        {'name': 'page',    'type': 'string'},
        {'name': 'ts',      'type': 'double'},
        {'name': 'session_id', 'type': ['null', 'string'], 'default': None},  # nuevo
    ],
}

events_v2 = [
    {'user_id': 'u1', 'page': '/foo', 'ts': 1.0, 'session_id': 's1'},
    {'user_id': 'u2', 'page': '/bar', 'ts': 2.0, 'session_id': None},
]

buf = BytesIO()
fastavro.writer(buf, schema_v2, events_v2)
data = buf.getvalue()
print(f'2 records v2: {len(data)} bytes')

# Consumer viejo (con schema v1) puede leer data v2 — los campos nuevos se ignoran
buf.seek(0)
reader = fastavro.reader(buf, reader_schema=schema_v1)
for r in reader:
    print('leído con schema v1:', r)
print('\n→ schema evolution backward-compatible OK')

## Ejercicio guiado

1. Bajá 1 año de NYC Taxi (CSV original). Convertí a Parquet con cada compresión. Reporte tamaño + tiempo de query.
2. Sobre el Parquet final, particioná por `pickup_date` (`partitionBy`). Compará tiempo de `WHERE pickup_date='X'` con/sin particionado.
3. Hacé un cambio NO compatible en Avro (renombrar `page` → `url` sin alias). Confirmá que rompe el consumer viejo.
4. Investigá Delta Lake o Iceberg sobre tu Parquet — agregá ACID + time travel.
5. Estimá ahorro de S3 al migrar CSV → Parquet zstd en tu workload real.

## Conclusiones

- Parquet es 5-20× más chico que CSV y 10-100× más rápido en queries analíticas (column pruning + predicate pushdown).
- Compresión zstd domina ratio; snappy domina speed.
- Avro brilla en streaming + schema evolution; Parquet brilla en storage + queries.
- CSV solo sobrevive para interop humano y datasets triviales.

## ✅ Soluciones de los ejercicios

Trabajamos con parquet real (pyarrow + DuckDB, ambos instalados). `fastavro` **no** está en
el laboratorio, así que el roundtrip Avro lo hacemos con un **codec binario schema-based
propio** que ilustra por qué un formato con schema pesa mucho menos que JSON. Todo sobre
datos sintéticos, sin internet.

### Ejercicio 1 — CSV → Parquet: tamaño y velocidad

Mismo dataset en CSV y en Parquet (snappy). Parquet es columnar + comprimido: pesa menos y
las queries que tocan pocas columnas son más rápidas.

In [ ]:
import tempfile, time
from pathlib import Path
import numpy as np, pandas as pd, duckdb
import pyarrow as pa, pyarrow.parquet as pq

WORK = Path(tempfile.gettempdir()) / "colformat_sim"; WORK.mkdir(exist_ok=True)
rng = np.random.default_rng(0)
N = 200_000
df = pd.DataFrame({
    "borough": rng.choice(["Manhattan", "Brooklyn", "Queens", "Bronx"], N),
    "pickup_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 31, N), unit="D"),
    "passengers": rng.integers(1, 5, N),
    "fare": rng.gamma(3.0, 4.0, N).round(2),
})
csv_p, pq_p = WORK / "trips.csv", WORK / "trips.parquet"
df.to_csv(csv_p, index=False)
df.to_parquet(pq_p, compression="snappy")

csv_mb, pq_mb = csv_p.stat().st_size/1e6, pq_p.stat().st_size/1e6
print(f"CSV: {csv_mb:.2f} MB | Parquet snappy: {pq_mb:.2f} MB | ratio {csv_mb/pq_mb:.1f}x")

n_manh = duckdb.sql(f"SELECT COUNT(*) FROM '{pq_p.as_posix()}' WHERE borough='Manhattan'").fetchone()[0]
assert pq_mb < csv_mb, "Parquet comprimido pesa menos que CSV"
assert n_manh == int((df["borough"] == "Manhattan").sum())
print("OK ejercicio 1 — Parquet más chico que CSV; query columnar correcta")

### Ejercicio 2 — Benchmark de compresión

Escribimos el mismo parquet con distintos codecs y comparamos tamaño + tiempo de lectura.
`zstd` suele ganar en ratio; `snappy` en velocidad. (Codecs no disponibles se saltan.)

In [ ]:
table = pa.Table.from_pandas(df)
rows = []
for codec in ["snappy", "zstd", "gzip", "lz4"]:
    p = WORK / f"trips_{codec}.parquet"
    try:
        pq.write_table(table, p, compression=codec)
    except Exception as e:      # codec no compilado en este pyarrow
        print(f"  {codec}: no disponible ({type(e).__name__})"); continue
    t0 = time.perf_counter(); _ = pq.read_table(p); t_read = time.perf_counter() - t0
    rows.append({"codec": codec, "MB": p.stat().st_size/1e6, "read_ms": t_read*1000})

bench = pd.DataFrame(rows).sort_values("MB")
print(bench.to_string(index=False))

assert len(bench) >= 2, "al menos snappy y gzip deberían estar disponibles"
assert bench["MB"].min() > 0
print("OK ejercicio 2 — benchmark de compresión (tamaño vs velocidad de lectura)")

### Ejercicio 3 — Predicate pushdown en DuckDB

Con `EXPLAIN ANALYZE`, comparamos las filas leídas de un `SELECT ... WHERE date=...` (usa
stats de row groups para saltar bloques) contra un scan completo.

In [ ]:
full_plan = duckdb.sql(f"EXPLAIN ANALYZE SELECT * FROM '{pq_p.as_posix()}'").fetchall()
where_plan = duckdb.sql(
    f"EXPLAIN ANALYZE SELECT * FROM '{pq_p.as_posix()}' WHERE pickup_date = DATE '2024-01-15'"
).fetchall()

# contamos filas reales devueltas por cada query como proxy de "rows read"
full_rows = duckdb.sql(f"SELECT COUNT(*) FROM '{pq_p.as_posix()}'").fetchone()[0]
where_rows = duckdb.sql(
    f"SELECT COUNT(*) FROM '{pq_p.as_posix()}' WHERE pickup_date = DATE '2024-01-15'"
).fetchone()[0]
print(f"scan completo: {full_rows:,} filas | con WHERE date: {where_rows:,} filas")
print("plan (fragmento):", str(where_plan)[:120], "...")

assert where_rows < full_rows, "el filtro por fecha reduce las filas materializadas"
assert full_rows == N
print("OK ejercicio 3 — el predicado reduce lo leído (pushdown con stats de parquet)")

### Ejercicio 4 — Row groups y estadísticas

Cada *row group* de un parquet guarda `min`/`max`/`null_count` por columna. Eso es lo que
permite el pushdown: si `max < valor_buscado`, se salta el bloque entero.

In [ ]:
# forzamos varios row groups para que haya stats por bloque
multi = WORK / "trips_rowgroups.parquet"
pq.write_table(table, multi, row_group_size=50_000)

pf = pq.ParquetFile(multi)
print("row groups:", pf.metadata.num_row_groups)
rg0 = pf.metadata.row_group(0)
fare_idx = [i for i in range(rg0.num_columns)
            if rg0.column(i).path_in_schema == "fare"][0]
stats = rg0.column(fare_idx).statistics
print(f"row group 0 -> fare min={stats.min:.2f} max={stats.max:.2f} nulls={stats.null_count}")

assert pf.metadata.num_row_groups >= 2, "el dataset se dividió en varios row groups"
assert stats.min <= stats.max
print("OK ejercicio 4 — stats min/max/null_count por row group inspeccionadas")

### Ejercicio 5 — Avro-like: schema + roundtrip vs JSON

Sin `fastavro`, implementamos un **codec binario row-oriented con schema fijo** (la idea de
Avro: el schema se conoce aparte, así los datos no repiten los nombres de campo). Serializamos
1000 eventos, deserializamos, y comparamos el tamaño contra JSON puro.

In [ ]:
import struct, json

# schema conocido (como el .avsc de Avro): campos y tipos, en orden
SCHEMA = [("user_id", "i"), ("page_id", "i"), ("ts", "q")]   # int, int, long

def avro_like_encode(events):
    buf = bytearray()
    for e in events:
        for name, fmt in SCHEMA:
            buf += struct.pack("<" + fmt, e[name])   # solo el valor, sin el nombre
    return bytes(buf)

def avro_like_decode(blob, n):
    size = struct.calcsize("<" + "".join(f for _, f in SCHEMA))
    out = []
    for i in range(n):
        vals = struct.unpack_from("<" + "".join(f for _, f in SCHEMA), blob, i*size)
        out.append({name: v for (name, _), v in zip(SCHEMA, vals)})
    return out

rng = np.random.default_rng(1)
events = [{"user_id": int(rng.integers(1, 1000)),
           "page_id": int(rng.integers(1, 50)),
           "ts": int(1_700_000_000 + i)} for i in range(1000)]

blob = avro_like_encode(events)
back = avro_like_decode(blob, len(events))
json_bytes = json.dumps(events).encode()

print(f"binario schema-based: {len(blob):,} bytes | JSON: {len(json_bytes):,} bytes "
      f"-> {len(json_bytes)/len(blob):.1f}x más chico")
assert back == events, "roundtrip exacto"
assert len(blob) < len(json_bytes), "el formato con schema pesa mucho menos que JSON"
print("OK ejercicio 5 — roundtrip schema-based (idea de Avro) mucho más compacto que JSON")